# Load dữ liệu

In [24]:
import pandas as pd

path = "./final-round.json"

# Nếu file là 1 list các object JSON
try:
    df = pd.read_json(path)
    print("Đọc JSON dạng array OK")
except ValueError:
    # Nếu là JSON Lines (mỗi dòng 1 object)
    df = pd.read_json(path, lines=True)
    print("Đọc JSON dạng lines OK")

df.head()

Đọc JSON dạng array OK


,review,sentiment
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,positive
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,positive
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,neutral
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,neutral
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,negative


# Hàm preprocessing: giữ như trước + bỏ dấu tiếng Việt

In [25]:
import re
import unicodedata

def clean_review_basic(text):
    """Tiền xử lý cơ bản như bạn yêu cầu: 
    lower, bỏ số, bỏ ký hiệu, thu gọn khoảng trắng
    """
    if not isinstance(text, str):
        return ""

    # viết thường
    text = text.lower()

    # bỏ số
    text = re.sub(r"\d+", " ", text)

    # bỏ ký hiệu, dấu câu (giữ lại chữ, số, khoảng trắng)
    # \w = chữ + số + _ ; \s = khoảng trắng
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # bỏ riêng dấu gạch dưới nếu còn
    text = text.replace("_", " ")

    # thu gọn khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def remove_vietnamese_accents(text):
    """Bỏ dấu tiếng Việt bằng unicodedata"""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    text = unicodedata.normalize('NFC', text)
    return text


def preprocess_review(text):
    # bước 1: clean cơ bản
    text = clean_review_basic(text)
    # bước 2: bỏ dấu tiếng Việt
    text = remove_vietnamese_accents(text)
    return text


In [26]:
df["review_clean"] = df["review"].apply(preprocess_review)
df[["review", "review_clean", "sentiment"]].head(10)

,review,review_clean,sentiment
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,tu hoc bong thay đoi cuoc đoi đen lop hoc tien...,positive
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,truong đh ton đuc thang cong bo nhom nghien cu...,positive
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,đe thi thu tot nghiep thpt mon toan cua thanh ...,neutral
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,so giao duc tphcm len tieng viec giao vien bi ...,neutral
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,nguoi me tran tro truoc gio ghi đon xanh nguye...,negative
5,"Tốt nghiệp loại giỏi dù không biết đọc viết, n...",tot nghiep loai gioi du khong biet đoc viet nu...,negative
6,Màn ‘hỏi xoáy’ bất ngờ của học sinh BRIS với g...,man hoi xoay bat ngo cua hoc sinh bris voi gia...,positive
7,Tổng Bí thư gợi mở miễn phí bữa trưa cho học s...,tong bi thu goi mo mien phi bua trua cho hoc s...,positive
8,Top 10 trường có điểm chuẩn lớp 10 cao nhất TP...,top truong co điem chuan lop cao nhat tphcm ho...,neutral
9,"Ai là người vẽ bản đồ tác chiến Xuân Lộc 1975,...",ai la nguoi ve ban đo tac chien xuan loc sau n...,positive


# Chuẩn hóa nhãn & encode label

In [27]:
# xem các nhãn hiện có
print(df["sentiment"].value_counts())

# chuẩn hóa về chữ thường
df["sentiment"] = df["sentiment"].str.lower().str.strip()

# mã hóa nhãn → số
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label_id"] = le.fit_transform(df["sentiment"])

print("Classes:", le.classes_)  # ví dụ: ['negative' 'neutral' 'positive']
df[["sentiment", "label_id"]].head()


sentiment
neutral     964
negative    756
positive    724
Name: count, dtype: int64
Classes: ['negative' 'neutral' 'positive']


,sentiment,label_id
0,positive,2
1,positive,2
2,neutral,1
3,neutral,1
4,negative,0


# Chia train / test (và optionally valid)

In [28]:
from sklearn.model_selection import train_test_split

X = df["review_clean"].values      # text đã preprocess + bỏ dấu
y = df["label_id"].values          # nhãn dạng số (0,1,2,...)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Vector hóa text bằng TF-IDF

In [29]:
# Import các thư viện cần thiết cho mô hình và tìm kiếm tham số
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# 1. Khởi tạo Pipeline
# Pipeline giúp quy trình Vector hóa -> Model liền mạch, tránh rò rỉ dữ liệu (data leakage)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),   # Bước 1: Vector hóa
    ('nb', MultinomialNB())         # Bước 2: Mô hình Naive Bayes (Multinomial phù hợp đếm từ)
])

# 2. Thiết lập lưới tham số (Parameter Grid) để tinh chỉnh
# Chúng ta sẽ thử nghiệm các thông số khác nhau để xem cái nào tốt nhất
param_grid = {
    # 'tfidf__ngram_range': Thử nghiệm từ đơn (1,1) hoặc cụm 2 từ (1,2)
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    
    # 'tfidf__max_df': Bỏ qua các từ xuất hiện quá thường xuyên (như stop words tự nhiên)
    'tfidf__max_df': [0.75, 1.0],
    
    # 'nb__alpha': Tham số làm mịn (smoothing) cho Naive Bayes
    'nb__alpha': [0.1, 0.5, 1.0]
}

# 3. Khởi tạo GridSearchCV
# cv=5: Chia tập train thành 5 phần để kiểm tra chéo (Cross Validation)
print("Đang huấn luyện và tìm kiếm tham số tốt nhất (Grid Search)...")
grid = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1,  # Sử dụng tất cả các nhân CPU để chạy nhanh hơn
    verbose=1
)

Đang huấn luyện và tìm kiếm tham số tốt nhất (Grid Search)...


In [30]:
grid.fit(X_train, y_train)

print("Best F1-macro (CV):", grid.best_score_)
print("\nBest params:")
for k, v in grid.best_params_.items():
    print(f"{k}: {v}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best F1-macro (CV): 0.7048593350383632

Best params:
nb__alpha: 0.1
tfidf__max_df: 0.75
tfidf__ngram_range: (1, 2)


In [31]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report (test):")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 0.689161554192229

Classification report (test):
              precision    recall  f1-score   support

           0       0.75      0.72      0.74       151
           1       0.65      0.75      0.70       193
           2       0.69      0.57      0.63       145

    accuracy                           0.69       489
   macro avg       0.70      0.68      0.69       489
weighted avg       0.69      0.69      0.69       489


Confusion matrix:
[[109  34   8]
 [ 19 145  29]
 [ 17  45  83]]


In [32]:
target_names = le.inverse_transform(sorted(df["label_id"].unique()))
print(classification_report(y_test, y_pred, target_names=target_names))

              precision    recall  f1-score   support

    negative       0.75      0.72      0.74       151
     neutral       0.65      0.75      0.70       193
    positive       0.69      0.57      0.63       145

    accuracy                           0.69       489
   macro avg       0.70      0.68      0.69       489
weighted avg       0.69      0.69      0.69       489

